# AuthentiScan — A3 session 5 of 6

**First run of this notebook?** Kaggle Secrets attach per notebook: open Add-ons →
Secrets and toggle `GITHUB_PAT` ON for this notebook first, or cell 1 fails with
'No user secrets exist'. One-time per session notebook.

Runs: **vgg19_ft** | worst-case (30 epochs) ~8.0 h | per-run time budget 460 min

Plan: `sem8_major/implementation_plan.md` §7a A3. Each config runs as its own
subprocess and a failure does not stop the session. Before running, check the
quota page shows enough GPU hours left for the worst case above.

**After it finishes:** download `results_a3_s5.zip` from the Output tab and send
it back — rows are merged into the repo with `code/merge_runs.py`, never retyped.


In [ ]:
# 1. Code: clone the private repo (Kaggle Secret GITHUB_PAT) or pull if already there
import os, subprocess
from kaggle_secrets import UserSecretsClient

PAT = UserSecretsClient().get_secret("GITHUB_PAT")
REPO_DIR = "/kaggle/working/sm7"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone",
                    f"https://{PAT}@github.com/rohityaduvxnshi/sm7.git", REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(f"{REPO_DIR}/sem8_major")
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)


In [ ]:
# 2. Extras only - never upgrade Kaggle's torch/torchvision (CUDA build is matched)
!pip install -q timm grad-cam

# Accelerator consistency: every matrix run must use the SAME GPU, or Paper 2's
# training-time comparison across architectures compares hardware, not models.
# Calibration and A2 ran on T4. HARD abort on anything else: an aborted assignment
# costs seconds; a P100 run would poison the training-time table. Relaunch (or
# re-push) until Kaggle assigns a T4, or pin it in Settings -> Accelerator.
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("GPU:", gpu)
assert "T4" in gpu, f"wrong accelerator: {gpu} - matrix runs are T4-only (plan 7a A3)"

# Dataset root autodetect: Kaggle moved dataset mounts (classic /kaggle/input/<slug>
# vs the namespaced /kaggle/input/datasets/<owner>/<slug> seen 22 Aug 2026 in batch
# runs), so never hardcode the path - find the dir that holds train/REAL.
import os
DATA = None
for root, dirs, _ in os.walk("/kaggle/input"):
    if "train" in dirs and os.path.isdir(os.path.join(root, "train", "REAL")):
        DATA = root
        break
    if root.count(os.sep) > 5:   # don't descend into the dataset's image folders
        dirs.clear()
assert DATA, "CIFAKE not found anywhere under /kaggle/input - is the dataset attached?"
print("CIFAKE mounted at:", DATA)


In [ ]:
# 3. Pre-flight: matrix configs must be in matrix state, or a subset run could
# silently produce a wrong row in the Paper 2 results table (plan 7a A3).
import yaml, pathlib
CONFIGS = ['vgg19_ft']
for name in CONFIGS:
    p = pathlib.Path(f"configs/{name}.yaml")
    c = yaml.safe_load(p.read_text(encoding="utf-8"))
    assert c.get("smoke_subset") is None, f"{p} has smoke_subset set"
    assert c["output"]["runs_csv"].endswith("runs.csv"), p
    print(f"OK {name}: {c['model']} {c['mode']} {c['optimizer']} lr={c['lr']} "
          f"bs={c['batch_size']} max_epochs={c['max_epochs']}")


In [ ]:
# 4. The session. One subprocess per config; a failure does not stop the rest.
# {DATA} is the autodetected mount from cell 2 (IPython interpolates python vars).
!python code/run_session.py configs/vgg19_ft.yaml \
    --data-root "{DATA}" --results-dir /kaggle/working/results


In [ ]:
# 5. What this session produced
import pandas as pd
df = pd.read_csv("/kaggle/working/results/runs.csv")
cols = ["run_id", "model", "mode", "best_epoch", "stop_reason", "train_time_min",
        "val_acc", "test_acc", "test_auc"]
display(df[cols])
print("\nSanity (gate G3): fine-tuned CNN val_acc should be mid-90s; anything near")
print("0.50 means a pipeline bug - stop and debug rather than spending more quota.")


In [ ]:
# 6. Package for download: results (incl. checkpoints) + nothing else
!cd /kaggle/working && zip -qr results_a3_s5.zip results && ls -lh results_a3_s5.zip
